## Notebook 03: Feature Engineering & Rule-Based Heuristic Baseline

---

### 1. Architectural Role & Project Context
Raw sensory and predictive inputs must be synthesized into domain-meaningful decision metrics that encapsulate disaster risk reduction (DRR) concepts. Additionally, before deploying machine learning models, rigorous engineering requires establishing a transparent, deterministic **Rule-Based Baseline** derived from standard operating procedures (SOPs).

This notebook:
1. Derives non-linear interaction features and operational strain indices via `src/component04_ml/feature_engineering.py`.
2. Evaluates the deterministic benchmark via `src/component04_ml/rule_baseline.py` on the identical held-out test partition.
3. Identifies the performance ceiling and failure modes of traditional rule-based systems to justify machine learning.

---

### 2. Linked Source Scripts (`src/component04_ml/`)
- **`src/component04_ml/feature_engineering.py`**:
  - `engineer_decision_features(df)`: Computes domain interaction indices:
    - `risk_exposure_index`: $P(\text{contam}) \times \log_{10}(\text{population})$
    - `infrastructure_hazard_index`: $P(\text{contam}) \times W_{\text{infra}}$ (Critical=1.0, Major=0.7, Standard=0.4, Minor=0.15)
    - `telemetry_staleness_index`: Normalized combination of LoRa packet latency and sensor freshness
    - `rainfall_runoff_hazard`: Runoff washout amplification index
    - `resource_bottleneck_flag`: High hazard ($P \ge 0.5$) with $\le 1$ operational response resource
- **`src/component04_ml/rule_baseline.py`**:
  - `RuleBasedPrioritizer`: Implements multi-criteria weighted scoring and expert override rules defined in `config/decision_rules.yaml`.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.component04_ml.feature_engineering import engineer_decision_features
from src.component04_ml.rule_baseline import RuleBasedPrioritizer
from sklearn.metrics import classification_report, f1_score

train_df = pd.read_csv(root_dir / "data" / "processed" / "train_scenarios.csv")
test_df = pd.read_csv(root_dir / "data" / "processed" / "test_scenarios.csv")
print(f"Loaded Train: {len(train_df)}, Test: {len(test_df)}")


Loaded Train: 3000, Test: 1000


### Section 1: Derive Engineered Decision Metrics
**Source Module:** `src/component04_ml/feature_engineering.py` -> `engineer_decision_features()`

#### Feature Definitions & Physical Interpretations:
1. **Socio-Hydrological Exposure Index (`risk_exposure_index`)**:
   $$\text{REI} = P(\text{contamination}) \times \log_{10}(\max(10, \text{exposed\_population}))$$
   Scales hazard severity logarithmically with human exposure to avoid skewing while magnifying large municipal threats.
2. **Infrastructure Hazard Index (`infrastructure_hazard_index`)**:
   Weights hazard by asset criticality (Drinking Water Intakes = 1.00, Hospitals = 0.70, Standard River Reaches = 0.40, Minor Ditches = 0.15).
3. **Telemetry Staleness Index (`telemetry_staleness_index`)**:
   Combines LoRa communication delay and sensor data age:
   $$\text{TSI} = 0.5 \times \min\left(1.0, \frac{\text{delay}}{3600}\right) + 0.5 \times \min\left(1.0, \frac{\text{freshness}}{7200}\right)$$
4. **Rainfall Runoff Hazard (`rainfall_runoff_hazard`)**:
   Multiplies hazard probability by recent rainfall normalized to a 50mm storm event.
5. **Resource Bottleneck Flag (`resource_bottleneck_flag`)**:
   Binary indicator triggering when hazard is acute ($P \ge 0.50$) and available resources (alternative water, inspection teams, treatment supplies) total $\le 1$.


In [2]:
train_eng = engineer_decision_features(train_df)
print("Engineered feature columns added:")
eng_cols = [
    "risk_exposure_index",
    "infrastructure_hazard_index",
    "telemetry_staleness_index",
    "rainfall_runoff_hazard",
    "total_resources_available",
    "resource_bottleneck_flag"
]
train_eng[eng_cols].head()


Engineered feature columns added:


,risk_exposure_index,infrastructure_hazard_index,telemetry_staleness_index,rainfall_runoff_hazard,total_resources_available,resource_bottleneck_flag
0,2.4906,0.4938,0.0801,0.0000,1,1
1,0.5434,0.0537,0.0189,0.0000,3,0
2,1.0322,0.2604,0.8538,0.0240,1,0
3,0.5500,0.0693,0.0321,0.0000,2,0
4,1.7144,0.5521,0.1331,0.0519,1,1


### Section 2: Execute Rule-Based Prioritization Baseline
**Source Module:** `src/component04_ml/rule_baseline.py` -> `RuleBasedPrioritizer`

#### Heuristic Scoring Framework:
The baseline evaluates scenarios against expert-defined weights loaded from `config/decision_rules.yaml`:
$$\text{Score} = w_c S_{\text{contam}} + w_p S_{\text{pop}} + w_i S_{\text{infra}} + w_e S_{\text{env}} + w_q S_{\text{quality}}$$
- Thresholds: Score $\ge 70 \implies$ **P1**, $\ge 50 \implies$ **P2**, $\ge 30 \implies$ **P3**, else **P4**.
- Safety Overrides: Automated escalation to P1 if $P(\text{contam}) \ge 0.85$ at a `CRITICAL` water intake.


In [3]:
baseline = RuleBasedPrioritizer()
baseline_preds = baseline.predict(test_df)
test_actuals = test_df["priority_level"].tolist()

print("Rule-Based Baseline Classification Report (Test Set):")
print(classification_report(test_actuals, baseline_preds, target_names=["P1", "P2", "P3", "P4"]))


Rule-Based Baseline Classification Report (Test Set):
              precision    recall  f1-score   support

          P1       1.00      0.39      0.56        77
          P2       0.73      0.33      0.45       398
          P3       0.53      0.71      0.60       424
          P4       0.45      1.00      0.62       101

    accuracy                           0.56      1000
   macro avg       0.68      0.61      0.56      1000
weighted avg       0.64      0.56      0.54      1000



### Section 3: Baseline Performance Evaluation & Limitations

#### Operational Benchmark Results:
- Evaluates Rule-Based Baseline predictions against ground-truth labels on the held-out test partition (1,000 scenarios).
- **Core Limitations of Rule-Based Heuristics**:
  - Rigid threshold cutoffs fail to capture subtle multi-sensor trade-offs.
  - Linear weighted scoring struggles with high-dimensional compound events (e.g. moderate contamination + severe LoRa blackout + localized torrential rainfall).
  - Motivates the need for non-linear, tree-based machine learning classifiers (Random Forest & XGBoost).


In [4]:
macro_f1 = f1_score(test_actuals, baseline_preds, average="macro")
acc = (np.array(test_actuals) == np.array(baseline_preds)).mean()
print(f"Rule Baseline Accuracy: {acc * 100:.2f}%")
print(f"Rule Baseline Macro F1: {macro_f1:.4f}")

# Critical P1 Recall
p1_mask = np.array(test_actuals) == "P1"
p1_recall = (np.array(baseline_preds)[p1_mask] == "P1").mean()
print(f"Critical P1 Recall:     {p1_recall * 100:.2f}%")


Rule Baseline Accuracy: 56.00%
Rule Baseline Macro F1: 0.5585
Critical P1 Recall:     38.96%
